# Model 3 — Multiple Linear Regression (MLR), Cases 1–3

**Idea:** tomorrow's power at a block is a weighted sum of inputs plus a constant,
ŷ = β₀ + β₁x₁ + … + βₚxₚ, with β chosen by ordinary least squares (minimise Σ(y − ŷ)²).

One row = one 15-minute block b of day D+1. Inputs per case:

| Case | Inputs (all at the same block b) | Count |
|---|---|---|
| 1 — Historic PV | power on day D | 1 |
| 2 — + Historic weather | Case 1 + GHI, module temp, ambient temp, wind speed, wind direction on day D | 6 |
| 3 — + Future weather (oracle) | Case 2 + the same 5 weather values on day D+1 | 11 |

Case 3 uses tomorrow's *measured* weather, so it is an upper bound (a perfect weather forecast), not a real forecast.

Same data, split, scoring window and metrics as the two baseline notebooks.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

## 1. Settings

In [ ]:
# Works whether the notebook is opened from src/ or from the project folder.
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
CLEAN_CSV = ROOT / "Data" / "Clean" / "Winter_Clean.csv"
RESULTS = ROOT / "results"

CAPACITY_MW = 21.5
BAND_MW = 0.07 * CAPACITY_MW  # ±7% of capacity (GERC 2019 tolerance) = 1.505 MW

# Scoring window 07:00-18:15 (blocks 29-74): the plant never generates outside it.
DAY_FIRST_BLOCK, DAY_LAST_BLOCK = 29, 74

# Chronological split by target day (dates inclusive).
SPLITS = {
    "train": ("2025-10-15", "2026-01-14"),
    "val": ("2026-01-15", "2026-02-07"),
    "test": ("2026-02-08", "2026-02-14"),
}

# Short names for the CSV columns.
RENAME = {
    "Act. Power (MW)": "power",
    "W-GHI (W/m2)": "ghi",
    "W-MODT- (°C)": "modt",
    "W-AMBT (°C)": "ambt",
    "W-WSPD (m/s)": "wspd",
    "W-WDIR- (Degree)": "wdir",
}
WEATHER = ["ghi", "modt", "ambt", "wspd", "wdir"]

CASES = {
    "Case 1": ["power_D"],
    "Case 2": ["power_D"] + [f"{w}_D" for w in WEATHER],
    "Case 3": ["power_D"] + [f"{w}_D" for w in WEATHER] + [f"{w}_D+1" for w in WEATHER],
}

## 2. Load the clean data and label each day with its split

In [ ]:
df = pd.read_csv(CLEAN_CSV, parse_dates=["TimeStamp", "Date"]).rename(columns=RENAME)

df["split"] = None
for name, (start, end) in SPLITS.items():
    df.loc[df["Date"].between(start, end), "split"] = name
assert df["split"].notna().all(), "some dates fall outside every split"

df.groupby("split")["Date"].agg(["min", "max", "nunique"]).loc[["train", "val", "test"]]

## 3. Build the inputs

Each row is block b of the target day D+1. For every variable:
- `_D` = its value at the same block on the previous day (`groupby("Block").shift(1)`),
- `_D+1` = its value on the target day itself (only used in Case 3).

The target `y` is `power` on the target day. Persistence (`power_D`) is kept as the reference for the skill score.

In [ ]:
by_block = df.groupby("Block")
for col in ["power"] + WEATHER:
    df[f"{col}_D"] = by_block[col].shift(1)
for w in WEATHER:
    df[f"{w}_D+1"] = df[w]

# 15 Oct has no previous day, so it has no inputs.
df = df.dropna(subset=["power_D"]).reset_index(drop=True)

# Example: one row = 27 Jan 2026, 11:15 (block 46).
df.loc[(df["Date"] == "2026-01-27") & (df["Block"] == 46), ["Date", "Block", "power"] + CASES["Case 3"]].T

## 4. Keep daytime rows

The model is trained and scored only on 07:00–18:15. Night is always 0 MW, so night forecasts are simply set to 0.

In [ ]:
daytime = df[df["Block"].between(DAY_FIRST_BLOCK, DAY_LAST_BLOCK)].copy()
train = daytime[daytime["split"] == "train"]

print("daytime rows per split:", daytime["split"].value_counts()[["train", "val", "test"]].to_dict())

## 5. Metrics

In [ ]:
def evaluate(actual, forecast):
    err = forecast - actual
    mae = err.abs().mean()
    rmse = np.sqrt((err**2).mean())
    # R² = 1 - SS_res / SS_tot: share of variation explained vs always predicting the mean.
    r2 = 1 - (err**2).sum() / ((actual - actual.mean()) ** 2).sum()
    return {
        "MAE (MW)": mae,
        "RMSE (MW)": rmse,
        "nMAE (%)": 100 * mae / CAPACITY_MW,
        "nRMSE (%)": 100 * rmse / CAPACITY_MW,
        "Within ±7% (%)": 100 * (err.abs() <= BAND_MW).mean(),
        "R² (%)": 100 * r2,
        "Blocks": len(err),
    }


def skill_score(rmse, rmse_persistence):
    # 0 = no better than persistence, positive = better, negative = worse.
    return 100 * (1 - rmse / rmse_persistence)

## 6. Fit one MLR per case

For each case: fit on the training daytime rows only, predict every daytime row, and clip forecasts to the physically possible range [0, 21.5] MW.
MLR has no hyperparameters, so validation is only used to score it, not to tune it.

In [ ]:
models = {}
for case, features in CASES.items():
    model = LinearRegression()
    model.fit(train[features], train["power"])
    daytime[case] = np.clip(model.predict(daytime[features]), 0, CAPACITY_MW)
    models[case] = model

## 7. Scores: every case on every split

In [ ]:
rows = []
for case in CASES:
    for split in ["train", "val", "test"]:
        part = daytime[daytime["split"] == split]
        scores = evaluate(part["power"], part[case])
        rmse_persistence = evaluate(part["power"], part["power_D"])["RMSE (MW)"]
        rows.append({
            "case": case,
            "split": split,
            **scores,
            "Skill vs persistence (%)": skill_score(scores["RMSE (MW)"], rmse_persistence),
        })

metrics = pd.DataFrame(rows).round(3)
metrics

Validation only, side by side (this is the table used to compare models):

In [ ]:
metrics[metrics["split"] == "val"].set_index("case").drop(columns=["split", "Blocks"])

## 8. Learned coefficients

Each value is βᵢ: how much the forecast (MW) changes when that input rises by 1 unit, others held fixed.
Units differ (MW, W/m², °C, m/s), so the sizes are not directly comparable across inputs.

In [ ]:
coefs = pd.DataFrame({
    case: pd.Series(np.r_[m.intercept_, m.coef_], index=["intercept"] + list(m.feature_names_in_))
    for case, m in models.items()
})
coefs.reindex(["intercept"] + CASES["Case 3"]).round(5)

## 9. Where does it fail? Worst validation days (Case 2, the realistic case)

In [ ]:
daily_mae = (
    daytime[daytime["split"] == "val"]
    .assign(abs_err=lambda x: (x["Case 2"] - x["power"]).abs())
    .groupby("Date")["abs_err"]
    .mean()
    .round(2)
)
daily_mae.sort_values(ascending=False).head(5)

In [ ]:
# Same three validation days as the baselines: clear, cloudy, clear.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, day in zip(axes, ["2026-01-25", "2026-01-26", "2026-01-27"]):
    d = daytime[daytime["Date"] == day]
    hours = (d["Block"] - 1) / 4
    ax.fill_between(hours, d["power"] - BAND_MW, d["power"] + BAND_MW, alpha=0.2, label="±7% band")
    ax.plot(hours, d["power"], color="black", label="Actual")
    ax.plot(hours, d["power_D"], ":", label="Persistence")
    for case in CASES:
        ax.plot(hours, d[case], "--", label=f"MLR {case}")
    ax.set_title(day)
    ax.set_xlabel("Hour of day")
axes[0].set_ylabel("Power (MW)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 10. Save forecasts and scores

In [ ]:
(RESULTS / "predictions").mkdir(parents=True, exist_ok=True)
(RESULTS / "metrics").mkdir(parents=True, exist_ok=True)

# Every block, with night forecasts set to 0.
out = df[["TimeStamp", "Date", "Block", "split", "power"]].rename(columns={"power": "actual"})
for case in CASES:
    out[f"MLR {case}"] = daytime[case].reindex(out.index).fillna(0.0)

out.to_csv(RESULTS / "predictions" / "mlr.csv", index=False)
metrics.to_csv(RESULTS / "metrics" / "mlr.csv", index=False)
coefs.to_csv(RESULTS / "metrics" / "mlr_coefficients.csv")